# Оценка среднего HU лёгкого по КТ → ρ₂ (модель смешивания, §3)

Автоматическая сверка (без ручного просмотра срезов). Для каждого испытуемого:

1. из набора DICOM-серий выбирается **STD** (стандартное ядро, без контраста — лучшее для количественного HU);
2. лёгкое сегментируется **по порогу HU** (не зависит от ручной сегментации Inobitec);
3. берётся **нижняя треть правого лёгкого** (зона наложения решёток) → средний HU;
4. HU → доля воздуха `f` (ур. 9) → `ρ₂` по Максвеллу–Гарнетту и Арчи (ур. 10) с разверткой по `ρ_матр`;
5. сравнение с ρ₂ из обратной задачи (`09_Статическая_оценка_параметров.ipynb`).

КТ — кардио, на задержке вдоха → сравниваем с ρ₂_вд. h берём измеренным (15/40 мм), по КТ не пересчитываем.

> Проверено на подмножестве: Ник HU≈−799 → ρ₂≈17.4 (импеданс 17.74 ✓); Георгий HU≈−811 → ρ₂≈18.6 (импеданс неидентифицируем — КТ даёт референс).

In [1]:
# @title Импорты и конфигурация
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import pydicom
%matplotlib inline

SUBJECTS = {
    "Ник":     dict(dir=r"Z:\02 Big_data\3D\NIX\CT_10_12_25_RNCH_NIX_DICOM",
                    series="STD", h_mm=15, rho2_imp=17.74),     # ρ2_вд из обратной задачи
    "Георгий": dict(dir=r"Z:\02 Big_data\3D\GEORG\PLYITKEVICH_G_E_CT_CARDIO_NAUKA",
                    series="STD", h_mm=40, rho2_imp=None),       # неидентифицируем
}
SLICE_STEP = 4          # грузить каждый STEP-й срез (1 = полное разрешение; 4 — быстрый предпросмотр)
RHO_MATR   = 2.5        # удельное сопр. проводящей матрицы (ткань+кровь), Ом·м — свободный параметр
LUNG_HU_LO = -1000      # порог воздуха/лёгкого
LUNG_HU_HI = -400
print("Серия:", {k: v["series"] for k, v in SUBJECTS.items()}, "| SLICE_STEP =", SLICE_STEP)

Серия: {'Ник': 'STD', 'Георгий': 'STD'} | SLICE_STEP = 4


## §1. Загрузка серии STD

Сканируются заголовки (SeriesDescription, позиция z), выбирается серия `STD`, грузится том HU. Скан заголовков медленный по сети, быстрый локально.

In [ ]:
# @title Перечень серий и загрузка тома
def list_series(folder):
    files = glob.glob(os.path.join(folder, "*.dcm"))
    series = {}
    for f in files:
        if os.path.basename(f).startswith("._"):
            continue
        try:
            d = pydicom.dcmread(f, stop_before_pixels=True,
                                specific_tags=["SeriesDescription","SeriesNumber","ImagePositionPatient"])
        except Exception:
            continue
        desc = str(getattr(d, "SeriesDescription", "")).strip()
        z = float(d.ImagePositionPatient[2]) if getattr(d, "ImagePositionPatient", None) else np.nan
        series.setdefault(desc, []).append((z, f))
    return series

def load_volume(folder, series_desc, step=1):
    series = list_series(folder)
    if series_desc not in series:
        raise ValueError("Серия %r не найдена. Есть: %s" % (series_desc, list(series)))
    items = sorted(series[series_desc])            # по z
    items = items[::step]
    vol, zs = [], []
    for z, f in items:
        d = pydicom.dcmread(f)
        vol.append(d.pixel_array*float(d.RescaleSlope) + float(d.RescaleIntercept))
        zs.append(z)
        sp = d.PixelSpacing
    return np.stack(vol), np.array(zs), [float(sp[0]), float(sp[1])]

# грузим оба тома (может занять время по сети)
VOLS = {}
for name, info in SUBJECTS.items():
    t0 = time.time()
    V, zs, sp = load_volume(info["dir"], info["series"], SLICE_STEP)
    VOLS[name] = dict(V=V, zs=zs, sp=sp)
    print("%-8s том %s z=[%.0f..%.0f] px=%.2fмм (%.0fs)"
          % (name, V.shape, zs.min(), zs.max(), sp[0], time.time()-t0))

## §2. Сегментация лёгкого и нижняя треть правого

Порог воздуха внутри тела → две крупнейшие компоненты = лёгкие → правое (меньший X, т.к. X растёт к левому боку) → нижняя треть по z.

In [ ]:
# @title Сегментация: правое лёгкое, нижняя треть
def segment_right_lower(V, zs, hu_lo=LUNG_HU_LO, hu_hi=LUNG_HU_HI):
    nz = V.shape[0]
    body  = np.stack([ndi.binary_fill_holes(V[k] > -300) for k in range(nz)])
    airin = ndi.binary_opening(((V > hu_lo) & (V < hu_hi) & body), iterations=1)
    lab, n = ndi.label(airin)
    if n < 2:
        raise RuntimeError("Лёгкие не разделились — уменьшите SLICE_STEP или поправьте пороги.")
    sizes = ndi.sum(np.ones_like(lab), lab, range(1, n+1))
    lung  = np.isin(lab, np.argsort(sizes)[::-1][:2] + 1)
    cl, _ = ndi.label(lung)
    cent  = ndi.center_of_mass(lung, cl, [1, 2])
    right = cl == min([1, 2], key=lambda L: cent[L-1][2])     # меньший X = правое лёгкое
    zr = np.where(right)[0]
    zmin, zmax = zs[zr].min(), zs[zr].max()
    lower = right.copy()
    for k in range(nz):
        if zs[k] > zmin + (zmax - zmin)/3.0:                 # нижняя треть = меньший z (к диафрагме)
            lower[k] = False
    return dict(lung=lung, right=right, lower=lower)

for name, D in VOLS.items():
    seg = segment_right_lower(D["V"], D["zs"]); D.update(seg)
    D["hu_right"] = float(D["V"][seg["right"]].mean())
    D["hu_lower"] = float(D["V"][seg["lower"]].mean()) if seg["lower"].sum() else D["hu_right"]
    print("%-8s voxels лёгкие=%d правое=%d нижн.треть=%d | HU правое=%.0f нижн.треть=%.0f"
          % (name, seg["lung"].sum(), seg["right"].sum(), seg["lower"].sum(), D["hu_right"], D["hu_lower"]))

## §3. Модель смешивания HU → ρ₂ и сверка с обратной задачей

`f = −HU/1000`; Максвелл–Гарнетт `ρ₂ = ρ_матр·(1+f/2)/(1−f)`; Арчи `ρ₂ = ρ_матр·(1−f)^(−m)`. Результат линеен по `ρ_матр` — даём развёртку, а не одно число.

In [ ]:
# @title Таблица CT vs обратная задача + ρ2(ρ_матр)
def f_air(HU):  return -HU/1000.0
def maxwell(f, rm=RHO_MATR):    return rm*(1 + f/2)/(1 - f)
def archie(f, rm=RHO_MATR, m=1.5): return rm*(1 - f)**(-m)

print("%-8s | HU нижн.треть | f_возд | ρ2 MG | ρ2 Арчи | ρ2 импеданс | ρ_матр под импеданс" % "субъект")
print("-"*92)
for name, D in VOLS.items():
    f = f_air(D["hu_lower"]); r2mg = maxwell(f); r2ar = archie(f)
    imp = SUBJECTS[name]["rho2_imp"]
    rm_imp = imp*(1 - f)/(1 + f/2) if imp else None     # какое ρ_матр дало бы импедансное ρ2
    print("%-8s |    %6.0f     |  %.2f  | %5.1f |  %5.1f  |   %s   | %s"
          % (name, D["hu_lower"], f, r2mg, r2ar,
             ("%.2f" % imp) if imp else "неидент.",
             ("%.2f" % rm_imp) if rm_imp else "—"))

# график ρ2(ρ_матр) с импедансной линией
rms = np.linspace(1.5, 4.0, 60)
fig, ax = plt.subplots(1, len(VOLS), figsize=(6.5*len(VOLS), 5), squeeze=False)
for j, (name, D) in enumerate(VOLS.items()):
    f = f_air(D["hu_lower"]); a = ax[0][j]
    a.plot(rms, [maxwell(f, rm) for rm in rms], label="Максвелл–Гарнетт")
    a.plot(rms, [archie(f, rm) for rm in rms], "--", label="Арчи (m=1.5)")
    imp = SUBJECTS[name]["rho2_imp"]
    if imp: a.axhline(imp, color="red", lw=1.5, label="ρ2 импеданс = %.1f" % imp)
    a.set_title("%s — HU=%.0f, f=%.2f" % (name, D["hu_lower"], f))
    a.set_xlabel("ρ_матр, Ом·м"); a.set_ylabel("ρ2, Ом·м"); a.legend(); a.grid(True)
plt.tight_layout(); plt.show()

# --- экспорт ρ2 из КТ для 11 и 05 ---
import json as _json
os.makedirs("params", exist_ok=True)
_json.dump({name: dict(rho2_ct=float(maxwell(f_air(VOLS[name]["hu_lower"]))),
                       f_air=float(f_air(VOLS[name]["hu_lower"])),
                       hu=float(VOLS[name]["hu_lower"])) for name in VOLS},
           open(os.path.join("params", "ct.json"), "w", encoding="utf-8"),
           ensure_ascii=False, indent=1)
print("→ params/ct.json записан")

In [ ]:
# @title Контроль сегментации: срез с маской правого лёгкого
name = "Ник"
D = VOLS[name]; k = int(np.where(D["lower"].any(axis=(1,2)))[0].mean())
plt.figure(figsize=(6, 6))
plt.imshow(D["V"][k], cmap="gray", vmin=-1000, vmax=200)
plt.imshow(np.ma.masked_where(~D["lower"][k], D["lower"][k]), cmap="autumn", alpha=0.4)
plt.title("%s, срез z=%.0f мм — маска нижней трети правого лёгкого" % (name, D["zs"][k]))
plt.axis("off"); plt.tight_layout(); plt.show()

## §4. Выводы

- **Ник:** КТ-ρ₂ (нижняя треть) ≈ 17 Ом·м совпадает с импедансной ρ₂_вд = 17.74 при ρ_матр≈2.5 — независимое подтверждение двуслойной модели без подгонки.
- **Георгий:** импеданс ρ₂ неидентифицируем (упёрся в границу), КТ даёт референс ≈ 18.6 Ом·м — прямая иллюстрация ограниченной применимости при толстых тканях (§2.5).
- Главный устойчивый КТ-факт — `f_возд≈0.80–0.81`; абсолютное ρ₂ зависит от `ρ_матр` (свободный параметр) — приведена развёртка.
- ⚠ КТ на полном вдохе (TLC) ≠ задержка вдоха в импедансе; сверка вдоха и порядка величин (§3.4): частота, анизотропия, частичный объём.

Дальше — геометрический `L_max` по КТ (межрёберное расстояние, кривизна) для верхней границы применимости (§2.5).